# Trained Never to Stop: reproducing the numbers

This notebook rebuilds the tables, Figure 2 and the Section 4-5 counts of our Gemma 4 Developer Agent Paper Track
writeup from the released run logs. It runs on a CPU in about a minute and needs no internet.

**What it does not rerun.** The expensive steps ran on rented GPUs (RunPod pods with one H100 or RTX PRO 6000):
- the base-model runs, the 242 restarts from failure points, and every evaluation, in the competition harness
  (swegemma 0.2.11, vLLM 0.19.1, the competition's 4-bit Gemma 4 12B);
- LoRA supervised fine-tuning, about 4 GPU-hours per arm, and DPO for arm (d), about 1 GPU-hour.

The code for those steps is in the project repository (`training/`, `runpod/`). Here we start from their outputs: graded
results (`task_results.jsonl`) and full agent traces for every run, the training sets and the training logs.

**Setup.** Attach the dataset *Gemma 4 recovery fine-tuning: data and logs* to this notebook. It holds the logs
and, under `code/`, the two analysis scripts the paper cites (`analyze_eval.py`, `paper_analyses.py`).

In [ ]:
import glob, json, os, sys
import pandas as pd
import matplotlib.pyplot as plt

# The dataset folder: the one Kaggle input that has eval/ and code/ (or set GEMMA4_DATA for a local copy)
DATA = os.environ.get("GEMMA4_DATA") or next(
    p.rsplit("/code/runner/", 1)[0] for p in glob.glob("/kaggle/input/**/code/runner/paper_analyses.py", recursive=True))
sys.path.insert(0, f"{DATA}/code/runner")
import analyze_eval, paper_analyses
print(DATA); print(sorted(os.listdir(DATA)))

## Section 3: the untrained model on each task set

Four base-model runs on each training task (FreshSWE-Py and our mined tasks) and five on OOD-45. Expected:
28.5% / 38.5%, 22% / 33%, 64% / 86%.

In [ ]:
def base_table(pattern, run_of):
    per = {}
    for f in glob.glob(f"{DATA}/{pattern}", recursive=True):
        for l in open(f):
            r = json.loads(l)
            per.setdefault(run_of(f), {})[r["instance_id"]] = bool(r["resolved"])
    tasks = set().union(*per.values())
    return {"tasks x runs": f"{len(tasks)} x {len(per)}",
            "solved per run": round(sum(sum(v.values()) for v in per.values()) / sum(len(v) for v in per.values()), 3),
            "solved in any run": round(sum(any(v.get(t) for v in per.values()) for t in tasks) / len(tasks), 3)}

run_r = lambda f: [p for p in f.split("/") if p[:1] in "rs" and p[1:].isdigit()][-1]
ood = {**{f"{DATA}/base_runs/ood45/{s}": s for s in ("s1", "s2", "s3")},
       f"{DATA}/eval/eval_v3/base/s2": "run1", f"{DATA}/eval/eval_v3_seed1/base/s1": "run2"}
t2 = {"FreshSWE-Py": base_table("base_runs/freshswe/*/fresh_base/r*/task_results.jsonl", run_r),
      "Our mined tasks": base_table("base_runs/mined/r*/task_results.jsonl", run_r)}
per = {name: paper_analyses.solved(d) for d, name in ood.items()}
tasks = set().union(*per.values())
t2["OOD-45"] = {"tasks x runs": f"{len(tasks)} x {len(per)}",
                "solved per run": round(sum(sum(v.values()) for v in per.values()) / sum(len(v) for v in per.values()), 3),
                "solved in any run": round(sum(any(v.get(t) for v in per.values()) for t in tasks) / len(tasks), 3)}
pd.DataFrame(t2).T

## Section 3: the two-model task-quality screen

Agreement between Claude Opus 5.5 and gpt-6.1-sol on which tasks to flag, per task set (the paper reports 94% overall,
kappa 0.82 to 0.89).

In [ ]:
for f in sorted(glob.glob(f"{DATA}/task_quality_screen/fairness_report_*.md")):
    for line in open(f):
        if line.startswith("| claude-opus") or line.startswith("Tasks flagged by"):
            print(os.path.basename(f).ljust(34), line.strip())

## Table 1 (Section 4) and the rest of the analysis

`paper_analyses.main` computes, from the evaluation folders:
- solved tasks per run and their mean (Table 1, including the round-2 arms: the second pass of (b) and the submit-only
  clean arm), and each agent against base, pairing runs of the same seed over both v3 runs, with a 95% paired bootstrap interval over tasks;
- churn: how many task outcomes change between two base runs, and between each agent and base;
- per-call behaviour (Table 2), solve times (Figure 2), the training data's reasoning length and the training loss.

In [ ]:
res = paper_analyses.main(DATA)
rows = []
for k, t in res["table3"].items():
    rows.append({"agent": k, "run 1 (seed 1)": t["solved"].get("1"), "run 2 (seed 2)": t["solved"].get("2"), "mean": t["mean"],
                 "vs base, points": None if "vs_base" not in t else round(100 * t["vs_base"], 1),
                 "95% interval": None if "ci95" not in t else [round(100 * x, 1) for x in t["ci95"]],
                 "seeds compared": t.get("vs_base_seeds")})
pd.DataFrame(rows).set_index("agent")


In [ ]:
print("Task outcomes that change between two base runs:", res["churn"].pop("base_vs_base"))
print("Base runs and tasks it never solved:", res["never"]["per_run_solved"], res["never"]["never"])
pd.DataFrame(res["churn"]).T.assign(changed=lambda d: d.gained + d.lost)

## Table 2: behaviour per model call (run 2 = seed 2)

Reasoning characters per model call and seconds per call as quartiles [Q1, median, Q3], and compactions per run.

In [ ]:
pd.DataFrame(res["table5"]).T

## Section 5 and Figure 1: how the runs ended, and the training data that taught it

The harness grades the workspace when time runs out, so a run can be solved without the agent ever submitting.
`training_pool` counts how the 228 solved FreshSWE-Py runs we trained on ended (expected: 88 submitted, 140 cut off at
the deadline; 21 of 24 nudges followed by another tool call; 1.1% of calls at the 4,096-token thinking cap).
`behaviour` gives the same measures for every evaluated agent and seed: submits and how many passed, solves at the
deadline, nudges and how many were followed by another tool call, the share of calls at the thinking cap, repeated
lines inside those capped thoughts, and generation speed.

In [ ]:
print(res["training_pool"])
pd.DataFrame(res["behaviour"]).T

## The signs show up early

Cap share and submits on five random 20-task samples of the seed-2 run: (a) and (c) hit the cap at three or more times
base's rate, and (b) submits less than base, in every sample.

In [ ]:
pd.concat({i: pd.DataFrame(s).T for i, s in enumerate(res["early_samples"])}, names=["sample", "agent"])

## Figure 1: how solved runs ended

Solved runs that called `submit_patch` against solved runs the harness found correct at the deadline: the solved FreshSWE-Py runs we trained on (expected 88 and 140), then each agent in the seed-2 run.

In [ ]:
names1 = {"base": "Base", "prompt": "Prompt-only", "a": "(a) Clean", "b": "(b) Recovery", "c": "(c) Mistake masked", "d": "(d) 17 pairs", "d_ablation": "(d) 23 pairs"}
tp, t3 = res["training_pool"], res["table3"]
rows = [("Runs we trained on", tp["submitted"], tp["solved"] - tp["submitted"])]
for k, n in names1.items():
    s = res["behaviour"][f"{k} seed2"]["submits_passed"]  # solved runs whose trace has a submit_patch call
    rows.append((n, s, t3[k]["solved"]["2"] - s))                # the rest were solved at the deadline
fig, ax = plt.subplots(figsize=(8, 4))
for i, (n, s, dl) in enumerate(rows[::-1]):
    tot = s + dl
    ax.barh(i, s / tot, color="tab:blue"); ax.barh(i, dl / tot, left=s / tot, color="0.8")
    ax.text(0.01, i, f"{100 * s / tot:.0f}%", va="center", fontsize=8, color="white")
    ax.text(1.01, i, f"{tot} solved", va="center", fontsize=8)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows[::-1]]); ax.set_xlim(0, 1.15); ax.set_xticks([])
ax.set_title("Share of solved runs that called submit_patch (blue) or were found correct at the deadline (grey)", fontsize=9)
plt.tight_layout(); plt.show()
pd.DataFrame(rows, columns=["agent", "solved with submit", "solved at the deadline"])

## Figure 3: where each agent's minutes go

Mean model minutes per task in the seed-2 run: thoughts that ran to the 4,096-token cap, the step after a compaction, the code-analysis sub-agent and everything else, against the 15-minute limit.

In [ ]:
mins = pd.DataFrame(res["minutes"]).T[["cap", "compaction", "subagent", "other"]].rename(index=names1)
ax = mins.plot.barh(stacked=True, figsize=(8, 4), color=["tab:blue", "0.55", "0.7", "0.85"])
ax.axvline(15, ls="--", color="0.4"); ax.text(15, -0.9, "15-minute limit", ha="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("model minutes per task (mean over 44 tasks)"); plt.tight_layout(); plt.show()
mins.assign(total=mins.sum(axis=1).round(1))

## Section 5: does more time, less thinking or a repetition penalty fix it?

Each changed setting against the same agent at 15 minutes on the same machine: the 30-minute limit ran on the seed-2 machine, the thinking budget of 1,024 tokens and the presence penalty of 0.4 on the seed-1 machine. `cap_share_pct` and `submits` are [15 minutes or default, changed].

In [ ]:
pd.DataFrame(res["settings_tests"]).T

## Figure 2: tasks solved by each minute of the run

The minute at which each solved run ended, from the agent clock in its trace (run 2 = seed 2, v3 build). The 15-minute limit is
checked between steps, so the last point (minute 16) holds runs that ended up to a minute late.

In [ ]:
names = {"base": "Base", "prompt": "Prompt-only", "a": "(a) Clean", "b": "(b) Recovery", "c": "(c) Mistakes hidden", "d": "(d) Preference"}
fig, ax = plt.subplots(figsize=(8, 4.5))
for k in names:
    c = res["curves"][k]
    ax.step(range(len(c)), c, where="post", lw=2.5 if k == "base" else 1.3,
            ls="--" if k == "prompt" else "-", color="tab:blue" if k == "base" else "0.6")
# end labels, pushed apart so none overlap
prev = 1e9
for k in sorted(names, key=lambda k: -res["curves"][k][-1]):
    y = min(res["curves"][k][-1], prev - 1.4); prev = y
    ax.text(16.3, y, f"{names[k]} {res['curves'][k][-1]}", va="center", fontsize=8, color="tab:blue" if k == "base" else "0.2")
ax.axvline(15, color="0.4", ls=":", lw=1); ax.text(14.9, 31.5, "15-minute limit", ha="right", fontsize=8)
ax.set_xticks(range(0, 16, 3)); ax.set_xlabel("Minutes since the run started"); ax.set_ylabel("Tasks solved (of 44)")
ax.set_title("Trained arms solve later: by minute 9 base has %d tasks, the trained arms %d to %d" % (
    res["curves"]["base"][9], min(res["curves"][k][9] for k in "abcd"), max(res["curves"][k][9] for k in "abcd")), fontsize=10)
ax.set_xlim(0, 20); ax.set_ylim(-0.5, 33); plt.tight_layout(); plt.show()


## Section 5: is it the training data? (and the training loss in Section 3)

Reasoning characters per trained step in the training examples (quartiles), clean (arm a) against splices (arms b, c),
and the training loss of each supervised arm (each batch is scored before the model trains on it).

In [ ]:
display(pd.DataFrame(res["train_data"]).T)
pd.DataFrame(res["train_loss"]).T

## Per-call numbers from the main script

`analyze_eval.py` gives each agent's outcomes and repeated failing calls (Section 4, "What training did change").

In [ ]:
rows = []
for root in ("eval/eval_v3", "eval/eval_v2_e2"):
    for d in sorted(glob.glob(f"{DATA}/{root}/*/s2")):
        s = analyze_eval.agent_summary(d)
        rows.append({"agent": f"{root.split('/')[-1]}/{d.split('/')[-2]}", "solved": s["solved"], **s["outcomes"],
                     "repeated failing calls per run": s["repeated_failing_calls_per_run"]})
pd.DataFrame(rows).set_index("agent").fillna(0)